In [1]:
!pip install osmnx geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.5 MB/s eta 0:00:00


In [2]:
from IPython.display import Javascript
from google.colab import output

def get_location():
    display(Javascript('''
        async function getLocation() {
            return new Promise((resolve, reject) => {
                navigator.geolocation.getCurrentPosition(
                    (position) => {
                        resolve({
                            lat: position.coords.latitude,
                            lon: position.coords.longitude
                        });
                    },
                    (error) => {
                        let errorMessage = "Geolocation error: ";
                        switch (error.code) {
                            case error.PERMISSION_DENIED:
                                errorMessage += "User denied the request for Geolocation.";
                                break;
                            case error.POSITION_UNAVAILABLE:
                                errorMessage += "Location information is unavailable.";
                                break;
                            case error.TIMEOUT:
                                errorMessage += "The request to get user location timed out.";
                                break;
                            default:
                                errorMessage += error.message || "An unknown error occurred.";
                        }
                        reject(errorMessage);
                    }
                );
            });
        }
    '''))

    return output.eval_js("getLocation()")

try:
    location = get_location()
    print("Your Live Location:", location)
except Exception as e:
    print(f"Failed to get location: {e}")

<IPython.core.display.Javascript object>

Failed to get location: Geolocation error: User denied the request for Geolocation.


In [3]:
import osmnx as ox
from geopy.distance import geodesic
import pandas as pd

def find_nearest_pharmacies_with_gmaps(user_lat, user_lon, radius=5000):

    tags = {"amenity": "pharmacy"}

    pharmacies = ox.features_from_point(
        (user_lat, user_lon),
        tags=tags,
        dist=radius
    )

    if pharmacies.empty:
        return "No pharmacies found nearby"

    # Filter for Point geometries and drop rows with missing data
    pharmacies = pharmacies[pharmacies.geometry.apply(lambda geom: geom.geom_type == 'Point')]
    pharmacies = pharmacies[["name", "geometry"]].dropna()

    if pharmacies.empty:
        return "No pharmacies found nearby with valid point locations."

    pharmacies["latitude"] = pharmacies.geometry.y
    pharmacies["longitude"] = pharmacies.geometry.x

    user_location = (user_lat, user_lon)

    pharmacies["distance_km"] = pharmacies.apply(
        lambda row: geodesic(
            user_location,
            (row["latitude"], row["longitude"])
        ).km,
        axis=1
    )

    pharmacies = pharmacies.sort_values("distance_km").head(5)

    # Create Google Maps link
    pharmacies["google_maps_link"] = pharmacies.apply(
        lambda row: f"https://www.google.com/maps/dir/{user_lat},{user_lon}/{row['latitude']},{row['longitude']}",
        axis=1
    )

    return pharmacies[["name", "distance_km", "google_maps_link"]]

In [4]:
user_lat = 37.4220
user_lon = -122.0841

find_nearest_pharmacies_with_gmaps(user_lat, user_lon)

name  distance_km  \
element id                                      
node    5802883723         NowRx     1.213300   
        4518607846  CVS Pharmacy     2.978071   
        2156900890  CVS Pharmacy     3.356203   
        1615026927  CVS Pharmacy     3.468896   
        433566460   CVS Pharmacy     3.663432   

                                                     google_maps_link  
element id                                                             
node    5802883723  https://www.google.com/maps/dir/37.422,-122.08...  
        4518607846  https://www.google.com/maps/dir/37.422,-122.08...  
        2156900890  https://www.google.com/maps/dir/37.422,-122.08...  
        1615026927  https://www.google.com/maps/dir/37.422,-122.08...  
        433566460   https://www.google.com/maps/dir/37.422,-122.08...